In [19]:
import numpy as np
import faiss
import pandas as pd
import sqlite3
import torch
from pathlib import Path
from itertools import combinations
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parents[1]

EMBED_PATH = ROOT / "data/embeddings/base_embeds.pt"
EMBED_NAME = EMBED_PATH.stem

IMAGE_DIR = ROOT / "images/ellipsoid" / EMBED_NAME
CACHE_DIR = ROOT / "data/cache" / EMBED_NAME
RESULTS_DIR = ROOT / "data/results" / EMBED_NAME

In [2]:
embeds = torch.load(EMBED_PATH, weights_only=False)
cls_tokens = embeds["cls_tokens"]

In [3]:
conn = sqlite3.connect(ROOT / "data/sql/metadata.db")

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [ ]:
category = "leather"

train_mask = meta["split"] == "train"
train_meta = meta[train_mask]
test_meta = meta[~train_mask]

train_cat_mask = train_meta["category"] == category

good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

train_emb = cls_tokens[train_mask]
cat_emb = train_emb[train_cat_mask]

test_emb = cls_tokens[~train_mask]
defect_test_emb = test_emb[defect_test_cat_mask]
good_test_emb = test_emb[good_test_cat_mask]

In [22]:
def fit_ellipsoid(covered_idx, cat_emb, reg=1e-4):
    X = cat_emb[covered_idx]
    center = X.mean(axis=0)

    cov = np.cov(X.T)

    # Regularise because local regions may have very few points
    cov += np.eye(cov.shape[0]) * reg

    cov_inv = np.linalg.pinv(cov)

    diff = X - center
    d2 = np.einsum("ij,jk,ik->i", diff, cov_inv, diff)

    threshold = d2.max() if len(d2) > 0 else 0.0

    return center, cov_inv, threshold

In [23]:
def ellipsoid_inside(X, center, cov_inv, threshold):
    diff = X - center
    d2 = np.einsum("ij,jk,ik->i", diff, cov_inv, diff)
    return d2 <= threshold

In [24]:
def clean_candidate(covered_idx, cat_emb, uncovered_mask, min_points=1, reg=1e-4):
    """
    Iteratively emoved embeddings form a candidate hypersphere until it statisifes the
    exclusive embedding embedding assignment constraint

    Starting from a candiate set of embeddings, the function repeatedly computed the
    centroid and radius of the hyphersphere. If the resulting hyphersphere contains embeddings that 
    have been assinged to previous hyperspheres, the embedding contributing most to the current radius
    is removed and the hypersphere is recomputed. This process continues until no previously assinged 
    embedings lie within the hypersphere or onbly a single embedding remains

    Parameters
    ----------
    covered_idx : ndarray
        Indices of embeddings current assigned to the candidate hypersphere

    cat_emb : ndarray
        Embedding matrix of the current object category

    uncovered_mask : ndarray
        Boolean mask indicating which embeddings have not yet been assigned
        to a hypersphere

    min_points : int
        The minimum anoubt of points a sphere should keep. Used to ensure previously
        accepted candidates are not rejected.
        
    Returns
    -------
    covered_idx : ndarray
        Indices of the cleaned hypersphere

    centroid : ndarray
        Centroid of the final hypersphere
    
    radius : float
        Radius of the final hypersphere. (Singleton spheres have radius 0)
    """

    while len(covered_idx) > min_points:
        center, cov_inv, threshold = fit_ellipsoid(covered_idx, cat_emb, reg=reg)

        inside_all = ellipsoid_inside(cat_emb, center, cov_inv, threshold)
        shared_count = np.sum(~uncovered_mask[inside_all])

        if shared_count == 0:
            return covered_idx, center, cov_inv, threshold
        
        # remove point furthest in Mahalanobis distance
        diff = cat_emb[covered_idx] - center
        d2_self = np.einsum("ij,jk,ik->i", diff, cov_inv, diff)
        worst_local = d2_self.argmax()

        covered_idx = np.delete(covered_idx, worst_local)

    center, cov_inv, threshold = fit_ellipsoid(covered_idx, cat_emb, reg=reg)
    return covered_idx, center, cov_inv, threshold

In [25]:
uncovered_mask = np.ones(len(cat_emb), dtype=bool)
ellipsoids = []

K_frac = 0.05

start_growth = 1.05
min_growth = 1

while uncovered_mask.any():
    uncovered_idx = np.where(uncovered_mask)[0]
    uncovered_emb = cat_emb[uncovered_idx].astype("float32")

    K = max(2, int(K_frac * len(uncovered_idx)))
    K = min(K, len(uncovered_idx) - 1)

    growth = max(min_growth, start_growth - 0.0025 * len(ellipsoids))

    if K < 1:
        covered_idx = uncovered_idx
        center, cov_inv, threshold = fit_ellipsoid(covered_idx, cat_emb)

    else:
        index = faiss.IndexFlatL2(uncovered_emb.shape[1])
        index.add(uncovered_emb)

        D, I = index.search(uncovered_emb.astype("float32"), k=K+1) # +1 as nearest is itself 

        D = D[:, 1:]    # Remove self
        I = I[:, 1:]

        avg_knn = D.mean(axis=1)

        local_compact  = avg_knn.argmin()
        neighbours_local = I[local_compact]

        full_covered_idx = uncovered_idx[np.r_[local_compact, neighbours_local]]

        covered_idx, center, cov_inv, threshold = clean_candidate(
            full_covered_idx,
            cat_emb,
            uncovered_mask
        )

        if len(full_covered_idx) == len(covered_idx):
            while True:
                old_covered_idx = covered_idx.copy()
                
                search_threshold = threshold * (growth ** 2)

                candidate_mask = ellipsoid_inside(cat_emb, center, cov_inv, search_threshold)
                full_new_covered_idx = np.where(candidate_mask & uncovered_mask)[0]
                
                new_covered_idx, new_centroid, new_radius, new_threshold = clean_candidate(
                    full_new_covered_idx, 
                    cat_emb, 
                    uncovered_mask, 
                    min_points=len(old_covered_idx)
                    ) 

                if len(new_covered_idx) > len(old_covered_idx):
                    covered_idx = new_covered_idx
                    centroid = new_centroid
                    radius = new_radius
                else:
                    break

            

    ellipsoids.append({
        "center": center,
        "cov_inv": cov_inv,
        "threshold": threshold,
        "covered_idx": covered_idx
    })

    uncovered_mask[covered_idx] = False

ellipsoids_df = pd.DataFrame([
    {
        "n_points": len(e["covered_idx"]),
        "threshold": e["threshold"]
    }
    for e in ellipsoids
])

ellipsoids_df

,n_points,threshold
0,11,9.090495
1,10,8.099688
2,10,8.099766
3,9,7.110888
4,9,7.110916
5,9,7.110868
6,8,6.124867
7,8,6.124880
8,7,5.142757
9,7,5.142767


In [26]:
overlaps = []

for i, j in combinations(range(len(ellipsoids)), 2):

    e1 = ellipsoids[i]
    e2 = ellipsoids[j]

    # Points owned by j inside ellipsoid i
    diff = cat_emb[e2["covered_idx"]] - e1["center"]
    d2 = np.einsum("ij,jk,ik->i", diff, e1["cov_inv"], diff)
    j_inside_i = np.sum(d2 <= e1["threshold"])

    # Points owned by i inside ellipsoid j
    diff = cat_emb[e1["covered_idx"]] - e2["center"]
    d2 = np.einsum("ij,jk,ik->i", diff, e2["cov_inv"], diff)
    i_inside_j = np.sum(d2 <= e2["threshold"])

    overlaps.append({
        "ellipsoid_i": i,
        "ellipsoid_j": j,
        "j_points_inside_i": j_inside_i,
        "i_points_inside_j": i_inside_j,
        "overlap": (j_inside_i + i_inside_j) > 0
    })

overlap_df = pd.DataFrame(overlaps)
overlap_df

,ellipsoid_i,ellipsoid_j,j_points_inside_i,i_points_inside_j,overlap
0,0,1,0,0,False
1,0,2,0,0,False
2,0,3,0,0,False
3,0,4,0,0,False
4,0,5,0,0,False
...,...,...,...,...,...
856,38,40,0,0,False
857,38,41,0,0,False
858,39,40,0,0,False
859,39,41,0,0,False


In [27]:
num_overlap = overlap_df["overlap"].sum()

num_overlap

np.int64(0)

In [28]:
def inside_any_count(X, ellipsoids):
    inside_any = np.zeros(len(X), dtype=bool)
    inside_count = np.zeros(len(X), dtype=int)

    for e in ellipsoids:
        diff = X - e["center"]
        d2 = np.einsum("ij,jk,ik->i", diff, e["cov_inv"], diff)

        inside = d2 <= e["threshold"]

        inside_any |= inside
        inside_count += inside

    return inside_any, inside_count

In [29]:

good_any, good_counts = inside_any_count(good_test_emb, ellipsoids)
defect_any, defect_counts = inside_any_count(defect_test_emb, ellipsoids)

print("Good accepted:", good_any.sum(), "/", len(good_test_emb))
print("Defect accepted:", defect_any.sum(), "/", len(defect_test_emb))

print("\nGood multi-ellipsoid:", np.sum(good_counts > 1))
print("Defect multi-ellipsoid:", np.sum(defect_counts > 1))

Good accepted: 0 / 20
Defect accepted: 0 / 63

Good multi-ellipsoid: 0
Defect multi-ellipsoid: 0


In [30]:
X = np.vstack([good_test_emb, defect_test_emb])

y_true = np.concatenate([
    np.zeros(len(good_test_emb)),
    np.ones(len(defect_test_emb))
])

scores = np.full(len(X), np.inf)

for e in ellipsoids:
    diff = X - e["center"]
    d2 = np.einsum("ij,jk,ik->i", diff, e["cov_inv"], diff)

    # <0 inside, >0 outside
    margin = d2 - e["threshold"]

    scores = np.minimum(scores, margin)

auroc = roc_auc_score(y_true, scores)
print("AUROC:", auroc)

AUROC: 0.9984126984126984
